# Sector-Rotation Training (Relative Targets)

Trains one LSTM per **(sector × horizon)** combination.

**Key change from previous run:** targets are now *relative* — a sector is labelled 1
if its forward return beats the median of all 8 sectors on the same day, 0 otherwise.
This gives ~50/50 class balance by construction, removing the positive market-drift
bias that inflated accuracy (and faked AUC) in the absolute-target run.

**Architecture:** same `SentimentLSTM(input=32, hidden=32, layers=2)` as per-stock.  
**Scheduler:** `ReduceLROnPlateau` — decays only when val_loss stops improving.

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import src
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("train_sector")

## Config

In [ ]:
from src.training import ComputeConfig, TrainingConfig

CUTOFF      = "2023-06-01"
VAL_FRAC    = 0.1
PRICE_YEARS = list(range(2018, 2025))
HORIZONS    = [5, 10, 21, 42]
WINDOW      = 20
SEED        = 42

config = TrainingConfig(
    window=WINDOW,
    batch_size=16,
    n_epochs=150,
    lr=1e-3,
    weight_decay=1e-4,
    patience=20,
    scheduler="plateau",
    scheduler_patience=10,
    grad_clip=1.0,
    seed=SEED,
)

compute_config = ComputeConfig(num_workers=0)
compute_config.setup()

print(f"Device    : {compute_config.device}")
print(f"Horizons  : {HORIZONS}")
print(f"Scheduler : {config.scheduler} (patience={config.scheduler_patience})")
print(f"Epochs    : {config.n_epochs}  early-stop patience={config.patience}")

## Load price and sentiment data

In [ ]:
from src.features.sectors import SECTORS
from src.repositories.prices import PriceRepository
from src.repositories.sentiment import SentimentRepository

PRICE_DIR = Path("../data/historical-prices/prices/data/historical-prices")
SENT_DIR  = Path("../data/sentiment/data/sentiment")

price_repo = PriceRepository(data_dir=PRICE_DIR)
sent_repo  = SentimentRepository(data_dir=SENT_DIR)

all_tickers = sorted({t for tickers in SECTORS.values() for t in tickers})

price_data:     dict[str, pd.DataFrame] = {}
sentiment_data: dict[str, pd.DataFrame] = {}

for ticker in all_tickers:
    try:
        price_data[ticker]     = price_repo.load_years(ticker, PRICE_YEARS)
        sentiment_data[ticker] = sent_repo.load(ticker)
    except FileNotFoundError:
        print(f"  Missing: {ticker}")

print(f"Loaded {len(price_data)} tickers")

## Pre-build sector price indices

Build each sector's equal-weight price index once and reuse it across all horizons.
This also lets us compute cross-sector relative labels before the training loop.

In [ ]:
from src.features.sectors import build_sector_price_index

price_indices: dict[str, pd.DataFrame] = {}
sector_tickers: dict[str, list[str]]   = {}

for sector_name, tickers in SECTORS.items():
    available = [t for t in tickers if t in price_data]
    if len(available) < 2:
        print(f"Skipping {sector_name}: only {len(available)} tickers")
        continue
    price_indices[sector_name]  = build_sector_price_index({t: price_data[t] for t in available})
    sector_tickers[sector_name] = available
    print(f"{sector_name:<20} {len(available)} tickers  {len(price_indices[sector_name])} trading days")

## Sanity check: class balance under relative targets

Each horizon should give ~50/50 positive rate across all sectors.
If any sector shows a strongly skewed rate, something is wrong with the label computation.

In [ ]:
from src.features.sectors import compute_cross_sector_labels

print("Positive-rate sanity check (should be ~0.50 everywhere)\n")
print(f"{'Sector':<20}  " + "  ".join(f"T+{h:2d}" for h in HORIZONS))
print("-" * 60)

for sector_name in price_indices:
    rates = []
    for horizon in HORIZONS:
        labels = compute_cross_sector_labels(price_indices, horizon)
        lbl    = labels[sector_name]
        valid  = lbl[lbl >= 0]
        rates.append(f"{valid.mean():.3f}")
    print(f"{sector_name:<20}  " + "  ".join(rates))

## Train: horizon × sector grid

Outer loop is **horizon** so cross-sector labels are computed once per horizon
and reused across all 8 sectors.

In [ ]:
from src.features.sectors import SectorDataset, build_sector_loaders
from src.model.lstm import SentimentLSTM
from src.model.trainer import Trainer
from src.repositories.models import ModelRepository

model_repo = ModelRepository()
records: list[dict] = []

for horizon in HORIZONS:
    print(f"\n{'#'*60}")
    print(f"# HORIZON = T+{horizon}")
    print(f"{'#'*60}")

    # Compute relative labels once for all sectors at this horizon
    cross_labels = compute_cross_sector_labels(price_indices, horizon)

    for sector_name in price_indices:
        print(f"\n{'='*60}")
        print(f"{sector_name}  |  horizon=T+{horizon}")
        print(f"{'='*60}")

        available    = sector_tickers[sector_name]
        sec_prices   = {t: price_data[t]     for t in available}
        sec_sent     = {t: sentiment_data[t] for t in available}

        try:
            ds = SectorDataset(
                name=sector_name,
                price_dfs=sec_prices,
                sentiment_dfs=sec_sent,
                window=WINDOW,
                horizon=horizon,
                target_labels=cross_labels[sector_name],
            )
        except RuntimeError as exc:
            print(f"  Dataset error: {exc}")
            continue

        train_loader, val_loader, test_loader = build_sector_loaders(
            ds, cutoff=CUTOFF, val_frac=VAL_FRAC, batch_size=config.batch_size,
        )

        n_train = len(train_loader.dataset)
        n_val   = len(val_loader.dataset)
        n_test  = len(test_loader.dataset)
        pos_rate = ds.y.mean()
        print(f"  Windows — train: {n_train}, val: {n_val}, test: {n_test}  pos_rate={pos_rate:.3f}")

        if n_train == 0 or n_test == 0:
            print("  Skipping: empty split")
            continue

        model = SentimentLSTM(
            n_factors=16, sentiment_dim=768, hidden_size=32, num_layers=2, dropout=0.2,
        )
        trainer      = Trainer(model, config, compute_config)
        train_result = trainer.fit(train_loader, val_loader)

        print(
            f"  Best epoch: {train_result.best_epoch} | "
            f"val_loss: {train_result.best_val_loss:.4f} | "
            f"val_auc: {train_result.best_val_auc:.4f}"
        )

        eval_result = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)
        print(
            f"  Test AUC: {eval_result.auc_mean:.3f} "
            f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
        )
        print(
            f"  Test Acc: {eval_result.accuracy_mean:.3f} "
            f"[{eval_result.accuracy_ci_low:.3f}, {eval_result.accuracy_ci_high:.3f}]"
        )

        model_repo.save(
            f"sector_rel_{sector_name}_T{horizon}",
            model,
            {
                "sector": sector_name, "tickers": available,
                "horizon": horizon, "window": WINDOW, "mode": "relative",
                "n_train": n_train, "n_val": n_val, "n_test": n_test,
                "best_epoch": train_result.best_epoch,
                "best_val_loss": train_result.best_val_loss,
                "best_val_auc": train_result.best_val_auc,
                "test_auc": eval_result.auc_mean,
                "test_accuracy": eval_result.accuracy_mean,
                "history": train_result.history,
            },
        )

        records.append({
            "sector":      sector_name,
            "horizon":     horizon,
            "n_train":     n_train,
            "n_test":      n_test,
            "best_epoch":  train_result.best_epoch,
            "val_auc":     train_result.best_val_auc,
            "test_auc":    eval_result.auc_mean,
            "auc_ci_low":  eval_result.auc_ci_low,
            "auc_ci_high": eval_result.auc_ci_high,
            "test_acc":    eval_result.accuracy_mean,
        })

print(f"\n\nDone. {len(records)} models trained.")

## Results

In [ ]:
df = pd.DataFrame(records)

auc_pivot = df.pivot(index="sector", columns="horizon", values="test_auc")
auc_pivot.columns = [f"T+{h}" for h in auc_pivot.columns]
auc_pivot["best"] = auc_pivot.max(axis=1)
auc_pivot = auc_pivot.sort_values("best", ascending=False)

print("=== Test AUC Matrix (relative targets) ===")
print(auc_pivot.to_string(float_format="%.3f"))
print(f"\nColumn means:")
print(auc_pivot.drop(columns="best").mean().to_string(float_format="%.3f"))
print(f"\nModels with AUC > 0.55: {(df['test_auc'] > 0.55).sum()} / {len(df)}")
print(f"Models with AUC > 0.50: {(df['test_auc'] > 0.50).sum()} / {len(df)}")

In [ ]:
# Full detail — epoch column tells us whether models actually trained
detail = df.sort_values("test_auc", ascending=False).copy()
detail["ci"] = detail.apply(
    lambda r: f"[{r['auc_ci_low']:.3f}, {r['auc_ci_high']:.3f}]", axis=1
)
print(detail[["sector", "horizon", "n_train", "n_test",
              "best_epoch", "val_auc", "test_auc", "ci", "test_acc"]]
      .to_string(index=False, float_format="%.3f"))

## Ablation: Technical Features Only (No Sentiment)

Re-runs the exact same training loop but zeros out all sentiment embeddings before
building the DataLoaders. The model architecture is identical — it still has a
`sentiment_proj` layer — but it always receives a zero vector, so the LSTM input
is effectively `[tech, zeros]`.

**If the ablation AUC ≈ full-model AUC**: the FinBERT embeddings contribute nothing.
The problem is at the data / signal level, not the model level.

**If full-model AUC > ablation AUC consistently**: sentiment is adding value and
the architecture is worth refining further.

In [ ]:
ablation_records: list[dict] = []

for horizon in HORIZONS:
    cross_labels = compute_cross_sector_labels(price_indices, horizon)

    for sector_name in price_indices:
        available  = sector_tickers[sector_name]
        sec_prices = {t: price_data[t]     for t in available}
        sec_sent   = {t: sentiment_data[t] for t in available}

        try:
            ds = SectorDataset(
                name=sector_name,
                price_dfs=sec_prices,
                sentiment_dfs=sec_sent,
                window=WINDOW,
                horizon=horizon,
                target_labels=cross_labels[sector_name],
            )
        except RuntimeError:
            continue

        # ── Zero out all sentiment embeddings ──────────────────────────
        import numpy as np
        ds.X_sent = np.zeros_like(ds.X_sent)
        # ───────────────────────────────────────────────────────────────

        train_loader, val_loader, test_loader = build_sector_loaders(
            ds, cutoff=CUTOFF, val_frac=VAL_FRAC, batch_size=config.batch_size,
        )
        if len(train_loader.dataset) == 0 or len(test_loader.dataset) == 0:
            continue

        model = SentimentLSTM(
            n_factors=16, sentiment_dim=768, hidden_size=32, num_layers=2, dropout=0.2,
        )
        trainer      = Trainer(model, config, compute_config)
        train_result = trainer.fit(train_loader, val_loader)
        eval_result  = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)

        print(
            f"{sector_name:<20} T+{horizon:2d} | "
            f"epoch={train_result.best_epoch:3d} | "
            f"AUC={eval_result.auc_mean:.3f} "
            f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
        )

        ablation_records.append({
            "sector":     sector_name,
            "horizon":    horizon,
            "best_epoch": train_result.best_epoch,
            "test_auc":   eval_result.auc_mean,
            "auc_ci_low": eval_result.auc_ci_low,
            "auc_ci_high":eval_result.auc_ci_high,
        })

print(f"\nDone. {len(ablation_records)} ablation models trained.")

In [ ]:
# Side-by-side comparison: full model vs tech-only ablation
abl = pd.DataFrame(ablation_records).rename(columns={"test_auc": "auc_techonly", "best_epoch": "epoch_techonly"})
full = pd.DataFrame(records)[["sector", "horizon", "best_epoch", "test_auc"]].rename(
    columns={"test_auc": "auc_full", "best_epoch": "epoch_full"}
)
cmp = full.merge(abl[["sector", "horizon", "auc_techonly", "epoch_techonly"]], on=["sector", "horizon"])
cmp["delta"] = cmp["auc_full"] - cmp["auc_techonly"]
cmp = cmp.sort_values("delta", ascending=False)

print("=== Full model vs Tech-only (delta = full − techonly) ===\n")
print(cmp[["sector", "horizon", "epoch_full", "auc_full",
           "epoch_techonly", "auc_techonly", "delta"]]
      .to_string(index=False, float_format="%.3f"))

print(f"\nMean delta (full − techonly): {cmp['delta'].mean():+.3f}")
print(f"Cases where full > techonly : {(cmp['delta'] > 0).sum()} / {len(cmp)}")
print(f"Cases where full > techonly by >0.02: {(cmp['delta'] > 0.02).sum()} / {len(cmp)}")

## Plan A — Scalar Sentiment Score as 17th Feature

**Hypothesis:** Instead of projecting 768-dim FinBERT embeddings (which has to compress
768 → 16 from ~1000 training windows), use `sentiment_score = 1.0·P(pos) + 0.5·P(neutral)` —
FinBERT's own directional summary already compressed into one number — appended
directly to the 16 technical features.

Changes vs the full model:
- `X_tech` shape: `(T, 17)` — column 17 is the aggregated daily sentiment score
- `X_sent` dummy: `(T, 1)` all zeros — the model's embedding path is disabled
- LSTM: `SentimentLSTM(n_factors=17, use_sentiment_proj=False)` — input size 17, no concat

**Plan B (score+delta)** follows with the 20-day rolling surprise:
`sentiment_score[t] − mean(sentiment_score[t−20:t])` appended as column 18.
Captures narrative shifts rather than the absolute sentiment level.
Both columns are min-max normalised per window by `_LazyDataset`, same as tech features.

In [ ]:
plan_a_records: list[dict] = []
plan_b_records: list[dict] = []

for sent_mode, result_list in [("score", plan_a_records), ("score+delta", plan_b_records)]:
    label = "Plan A (score)" if sent_mode == "score" else "Plan B (score+delta)"
    n_factors = 17 if sent_mode == "score" else 18
    print(f"\n{'#'*60}")
    print(f"# {label}  —  n_factors={n_factors}")
    print(f"{'#'*60}")

    for horizon in HORIZONS:
        cross_labels = compute_cross_sector_labels(price_indices, horizon)

        for sector_name in price_indices:
            available  = sector_tickers[sector_name]
            sec_prices = {t: price_data[t]     for t in available}
            sec_sent   = {t: sentiment_data[t] for t in available}

            try:
                ds = SectorDataset(
                    name=sector_name,
                    price_dfs=sec_prices,
                    sentiment_dfs=sec_sent,
                    window=WINDOW,
                    horizon=horizon,
                    target_labels=cross_labels[sector_name],
                    sentiment_mode=sent_mode,
                )
            except RuntimeError as exc:
                print(f"  {sector_name} T+{horizon}: Dataset error: {exc}")
                continue

            train_loader, val_loader, test_loader = build_sector_loaders(
                ds, cutoff=CUTOFF, val_frac=VAL_FRAC, batch_size=config.batch_size,
            )
            if len(train_loader.dataset) == 0 or len(test_loader.dataset) == 0:
                continue

            model = SentimentLSTM(
                n_factors=n_factors,
                sentiment_dim=1,          # dummy — ignored by model
                hidden_size=32,
                num_layers=2,
                dropout=0.2,
                use_sentiment_proj=False,  # tech-only LSTM, score already in X_tech
            )
            trainer      = Trainer(model, config, compute_config)
            train_result = trainer.fit(train_loader, val_loader)
            eval_result  = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)

            print(
                f"  {sector_name:<20} T+{horizon:2d} | "
                f"epoch={train_result.best_epoch:3d} | "
                f"AUC={eval_result.auc_mean:.3f} "
                f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
            )

            result_list.append({
                "sector":     sector_name,
                "horizon":    horizon,
                "best_epoch": train_result.best_epoch,
                "val_auc":    train_result.best_val_auc,
                "test_auc":   eval_result.auc_mean,
                "auc_ci_low": eval_result.auc_ci_low,
                "auc_ci_high":eval_result.auc_ci_high,
                "test_acc":   eval_result.accuracy_mean,
            })

print(f"\nPlan A: {len(plan_a_records)} models  |  Plan B: {len(plan_b_records)} models")

In [ ]:
# Four-way comparison: full embedding vs tech-only vs Plan A vs Plan B
abl_df = pd.DataFrame(ablation_records)[["sector", "horizon", "test_auc"]].rename(columns={"test_auc": "auc_techonly"})
pla_df = pd.DataFrame(plan_a_records)[["sector", "horizon", "test_auc", "best_epoch"]].rename(
    columns={"test_auc": "auc_planA", "best_epoch": "epoch_planA"})
plb_df = pd.DataFrame(plan_b_records)[["sector", "horizon", "test_auc", "best_epoch"]].rename(
    columns={"test_auc": "auc_planB", "best_epoch": "epoch_planB"})
base_df = pd.DataFrame(records)[["sector", "horizon", "best_epoch", "test_auc"]].rename(
    columns={"test_auc": "auc_emb768", "best_epoch": "epoch_emb768"})

cmp4 = base_df \
    .merge(abl_df,  on=["sector", "horizon"], how="left") \
    .merge(pla_df,  on=["sector", "horizon"], how="left") \
    .merge(plb_df,  on=["sector", "horizon"], how="left")

# Best AUC across all four approaches per row
cmp4["best_of_4"] = cmp4[["auc_emb768", "auc_techonly", "auc_planA", "auc_planB"]].max(axis=1)
cmp4["winner"]    = cmp4[["auc_emb768", "auc_techonly", "auc_planA", "auc_planB"]].idxmax(axis=1).str.replace("auc_", "")
cmp4 = cmp4.sort_values("auc_planA", ascending=False)

print("=== Four-way comparison (sorted by Plan A AUC) ===\n")
print(cmp4[["sector", "horizon", "epoch_emb768", "auc_emb768",
            "auc_techonly", "epoch_planA", "auc_planA",
            "epoch_planB", "auc_planB", "winner"]]
      .to_string(index=False, float_format="%.3f"))

print(f"\n{'─'*60}")
print(f"Mean AUC  —  emb768: {cmp4['auc_emb768'].mean():.3f}  "
      f"techonly: {cmp4['auc_techonly'].mean():.3f}  "
      f"planA: {cmp4['auc_planA'].mean():.3f}  "
      f"planB: {cmp4['auc_planB'].mean():.3f}")
print(f"Models > 0.55  —  emb768: {(cmp4['auc_emb768']>0.55).sum()}  "
      f"techonly: {(cmp4['auc_techonly']>0.55).sum()}  "
      f"planA: {(cmp4['auc_planA']>0.55).sum()}  "
      f"planB: {(cmp4['auc_planB']>0.55).sum()}")
print(f"\nWinner counts:\n{cmp4['winner'].value_counts().to_string()}")